# Fuzzy Logic Model for the BMWP/Col Water Quality Index

This notebook builds a Mamdani fuzzy inference system that predicts the **BMWP/Col** biological water quality index of the Cali River from five physicochemical predictors (BOD5, DO, turbidity, conductivity and TDS).

Membership functions for the predictors are defined from the literature/expert criteria, and the rule base is generated automatically: each observation in the dataset becomes one fuzzy rule. The model is then evaluated **in sample** (see the Limitations section).

## Data Loading

In [ ]:
import pandas as pd

# Cali River physicochemical + BMWP dataset (source: CVC)
file_path = "../../data/Database - BMWP.xlsx"
df = pd.read_excel(file_path)
df.columns = df.columns.str.strip()
df

In [ ]:
# Keep only the predictors and the BMWP response used by the model
columns_needed = ['OD', 'DBO5', 'SDT', 'Turbiedad', 'Conductividad', 'BMWP']
filtered_df = df[columns_needed]

print("First rows of the filtered dataset:")
display(filtered_df.head())
print("\nDescriptive statistics:")
display(filtered_df.describe())

### Outlier removal

Excessively high values are trimmed with an IQR rule. The upper bound uses a more tolerant margin (2.5 x IQR) to preserve variability given the small sample size.

In [ ]:
# IQR-based removal of excessively high values (tolerant upper bound)
def remove_excessive_outliers(df, column, margin=2.5):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR          # standard lower bound
    upper_bound = Q3 + margin * IQR       # tolerant upper bound
    return df[(df[column] >= lower_bound) & (df[column] <= upper_bound)]

filtered_df_cleaned = filtered_df.copy()
for column in ['DBO5', 'Turbiedad', 'Conductividad', 'OD', 'SDT']:
    filtered_df_cleaned = remove_excessive_outliers(filtered_df_cleaned, column, margin=2.5)

print(f"Original dataset size: {len(filtered_df)}")
print(f"Size after outlier removal: {len(filtered_df_cleaned)}")
print("\nDescriptive statistics of the clean dataset:")
print(filtered_df_cleaned.describe())

## Model Definition

The BMWP membership functions are based on the standard BMWP/Col quality classes found in the literature.

In [ ]:
import numpy as np
import skfuzzy as fuzz
from skfuzzy import control as ctrl

# Input variables (antecedents): universes and triangular membership functions
dbo5 = ctrl.Antecedent(np.arange(0, 9.1, 0.1), 'DBO5')
od = ctrl.Antecedent(np.arange(5, 7.6, 0.1), 'OD')
turbiedad = ctrl.Antecedent(np.arange(0, 31.1, 0.1), 'Turbiedad')
conductividad = ctrl.Antecedent(np.arange(50, 501, 1), 'Conductividad')
sdt = ctrl.Antecedent(np.arange(30, 131, 1), 'SDT')

# Biochemical Oxygen Demand (BOD5)
dbo5['Bajo'] = fuzz.trimf(dbo5.universe, [0, 2, 3])
dbo5['Medio'] = fuzz.trimf(dbo5.universe, [2.5, 4, 6])
dbo5['Alto'] = fuzz.trimf(dbo5.universe, [5, 7, 9])

# Dissolved Oxygen (DO)
od['Bajo'] = fuzz.trimf(od.universe, [4, 5, 5.8])
od['Medio'] = fuzz.trimf(od.universe, [5.5, 6.3, 6.9])
od['Alto'] = fuzz.trimf(od.universe, [6.5, 7, 7.5])

# Turbidity
turbiedad['Baja'] = fuzz.trimf(turbiedad.universe, [0, 2, 5])
turbiedad['Media'] = fuzz.trimf(turbiedad.universe, [4, 8, 12])
turbiedad['Alta'] = fuzz.trimf(turbiedad.universe, [10, 20, 31])

# Conductivity
conductividad['Baja'] = fuzz.trimf(conductividad.universe, [50, 80, 150])
conductividad['Media'] = fuzz.trimf(conductividad.universe, [100, 250, 400])
conductividad['Alta'] = fuzz.trimf(conductividad.universe, [300, 450, 500])

# Total Dissolved Solids (TDS)
sdt['Bajo'] = fuzz.trimf(sdt.universe, [30, 50, 70])
sdt['Medio'] = fuzz.trimf(sdt.universe, [60, 90, 110])
sdt['Alto'] = fuzz.trimf(sdt.universe, [100, 120, 130])

# Output variable (consequent): BMWP index, classes follow the BMWP/Col literature
bmwp = ctrl.Consequent(np.arange(0, 121, 1), 'bmwp')
bmwp['Muy crítica'] = fuzz.trimf(bmwp.universe, [0, 0, 15])
bmwp['Crítica'] = fuzz.trimf(bmwp.universe, [15, 35, 35])
bmwp['Dudosa'] = fuzz.trimf(bmwp.universe, [36, 60, 60])
bmwp['Aceptable'] = fuzz.trimf(bmwp.universe, [61, 100, 100])
bmwp['Buena'] = fuzz.trimf(bmwp.universe, [101, 120, 120])

## Visualisation of the BMWP membership functions

In [ ]:
import matplotlib.pyplot as plt

colors = {
    "Muy crítica": "red",
    "Crítica": "orange",
    "Dudosa": "yellow",
    "Aceptable": "green",
    "Buena": "blue",
}

plt.figure(figsize=(10, 6))
for label, color in colors.items():
    plt.plot(bmwp.universe, bmwp[label].mf, color=color, label=label, linewidth=2)

plt.title("BMWP Membership Functions", fontsize=16)
plt.xlabel("BMWP index", fontsize=14)
plt.ylabel("Membership", fontsize=14)
plt.legend(loc="upper left", fontsize=12)
plt.grid(True, linestyle="--", alpha=0.7)
plt.tight_layout()
plt.show()

## Predictor Categorisation

Each crisp value is mapped to its dominant linguistic category, so that every observation can be turned into a fuzzy rule.

In [ ]:
import skfuzzy as fuzz

# Generic helper: assign a crisp value to the linguistic category with the
# highest membership degree under the previously defined membership functions.
def categorize_variable(value, universe, mf_dict):
    memberships = {cat: fuzz.interp_membership(universe, mf, value)
                   for cat, mf in mf_dict.items()}
    return max(memberships, key=memberships.get)

def categorize_dbo5(value):
    return categorize_variable(value, dbo5.universe,
        {'Bajo': dbo5['Bajo'].mf, 'Medio': dbo5['Medio'].mf, 'Alto': dbo5['Alto'].mf})

def categorize_od(value):
    return categorize_variable(value, od.universe,
        {'Bajo': od['Bajo'].mf, 'Medio': od['Medio'].mf, 'Alto': od['Alto'].mf})

def categorize_turbiedad(value):
    return categorize_variable(value, turbiedad.universe,
        {'Baja': turbiedad['Baja'].mf, 'Media': turbiedad['Media'].mf, 'Alta': turbiedad['Alta'].mf})

def categorize_conductividad(value):
    return categorize_variable(value, conductividad.universe,
        {'Baja': conductividad['Baja'].mf, 'Media': conductividad['Media'].mf, 'Alta': conductividad['Alta'].mf})

def categorize_sdt(value):
    return categorize_variable(value, sdt.universe,
        {'Bajo': sdt['Bajo'].mf, 'Medio': sdt['Medio'].mf, 'Alto': sdt['Alto'].mf})

# Categorise the response variable too
def categorize_bmwp(value):
    return categorize_variable(value, bmwp.universe, {
        'Muy crítica': bmwp['Muy crítica'].mf,
        'Crítica': bmwp['Crítica'].mf,
        'Dudosa': bmwp['Dudosa'].mf,
        'Aceptable': bmwp['Aceptable'].mf,
        'Buena': bmwp['Buena'].mf,
    })

def categorize_row(row):
    return {
        'DBO5': categorize_dbo5(row['DBO5']),
        'OD': categorize_od(row['OD']),
        'Turbiedad': categorize_turbiedad(row['Turbiedad']),
        'Conductividad': categorize_conductividad(row['Conductividad']),
        'SDT': categorize_sdt(row['SDT']),
        'BMWP': categorize_bmwp(row['BMWP']),
    }

categorized_df = filtered_df_cleaned.apply(categorize_row, axis=1, result_type='expand')
print(categorized_df)

## Rule Generation

One fuzzy rule is created per observation: the antecedent is the conjunction of the categorised predictors and the consequent is the categorised BMWP class.

In [ ]:
rules = []
for _, row in categorized_df.iterrows():
    antecedent = (
        dbo5[row['DBO5']] &
        od[row['OD']] &
        turbiedad[row['Turbiedad']] &
        conductividad[row['Conductividad']] &
        sdt[row['SDT']]
    )
    consequent = bmwp[row['BMWP']]
    rules.append(ctrl.Rule(antecedent, consequent))

bmwp_ctrl = ctrl.ControlSystem(rules)
bmwp_sim = ctrl.ControlSystemSimulation(bmwp_ctrl)
print(f"{len(rules)} rules were generated for the fuzzy system.")

## Evaluation

The model is run on every observation used to build the rules (in-sample). The defuzzified output is re-categorised into a BMWP class and compared with the real class.

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

predictions = []
real_categories = categorized_df['BMWP']

for _, row in categorized_df.iterrows():
    try:
        bmwp_sim.input['DBO5'] = row['DBO5']
        bmwp_sim.input['OD'] = row['OD']
        bmwp_sim.input['Turbiedad'] = row['Turbiedad']
        bmwp_sim.input['Conductividad'] = row['Conductividad']
        bmwp_sim.input['SDT'] = row['SDT']
        bmwp_sim.compute()
        predictions.append(categorize_bmwp(bmwp_sim.output['bmwp']))
    except Exception as e:
        print(f"Error processing row: {row.to_dict()}, error: {e}")
        predictions.append("Unknown")

labels_present = list(set(real_categories) | set(predictions))
cm = confusion_matrix(real_categories, predictions, labels=labels_present)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', annot_kws={"size": 16},
            xticklabels=labels_present, yticklabels=labels_present)
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Real")
plt.show()

print("\nClassification report:")
print(classification_report(real_categories, predictions, labels=labels_present))

### Performance indicators

In [ ]:
from sklearn.metrics import cohen_kappa_score, roc_auc_score, accuracy_score

real_categories = real_categories.astype(str)
predictions = [str(p) for p in predictions]

# Correctly Classified Instances (overall accuracy)
cci = accuracy_score(real_categories, predictions)
print(f"Correctly Classified Instances (CCI): {cci:.4f}")

try:
    kappa = cohen_kappa_score(real_categories, predictions)
    print(f"Cohen's Kappa: {kappa:.4f}")
except ValueError as e:
    print(f"Could not compute Kappa: {e}")

cm = confusion_matrix(real_categories, predictions, labels=labels_present)
total = np.sum(cm)

# Weighted global sensitivity and simple-average global specificity
sensitivity_global = 0.0
specificity_sum = 0.0
classes_with_data = 0
per_class = {}
for i, label in enumerate(labels_present):
    tp = cm[i, i]
    fn = np.sum(cm[i, :]) - tp
    fp = np.sum(cm[:, i]) - tp
    tn = total - (tp + fn + fp)
    sens = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    sensitivity_global += sens * (tp + fn)
    if (tn + fp) > 0:
        specificity_sum += spec
        classes_with_data += 1
    per_class[label] = (sens, spec)

sensitivity_global /= total
specificity_global = specificity_sum / classes_with_data
print(f"\nGlobal sensitivity: {sensitivity_global:.4f}")
print(f"Global specificity: {specificity_global:.4f}")

print("\nPer-class sensitivity / specificity:")
for label, (sens, spec) in per_class.items():
    print(f"Class '{label}': sensitivity = {sens:.4f}, specificity = {spec:.4f}")

# Micro-averaged AUC over one-hot encoded labels
real_bin = pd.get_dummies(real_categories).reindex(columns=labels_present, fill_value=0)
pred_bin = pd.get_dummies(predictions).reindex(columns=labels_present, fill_value=0)
try:
    auc_micro = roc_auc_score(real_bin, pred_bin, average="micro", multi_class="ovo")
    print(f"\nArea Under the Curve (AUC, micro-average): {auc_micro:.4f}")
except ValueError as e:
    print(f"Could not compute AUC: {e}")

## Visualisation: ROC curve

In [ ]:
from sklearn.metrics import roc_curve, auc

fpr_micro, tpr_micro, _ = roc_curve(real_bin.values.ravel(), pred_bin.values.ravel())
roc_auc_micro = auc(fpr_micro, tpr_micro)

plt.figure(figsize=(8, 6))
plt.plot(fpr_micro, tpr_micro, color='blue', lw=2,
         label=f'Global ROC curve (AUC = {roc_auc_micro:.2f})')
plt.plot([0, 1], [0, 1], color='navy', linestyle='--', label='Baseline (AUC = 0.50)')
plt.xlabel('False Positive Rate (FPR)')
plt.ylabel('True Positive Rate (TPR)')
plt.title('Global ROC Curve - BMWP Fuzzy Model')
plt.legend(loc='lower right')
plt.grid()
plt.show()

## Limitations

Rules were derived from the same observations used for evaluation. Performance metrics therefore reflect in-sample fit and should not be interpreted as evidence of out-of-sample generalisation capacity.

A proper out-of-sample assessment (e.g. leave-one-out cross-validation) is a future improvement, not something implemented here.